# 點積與叉積 (Dot Product & Cross Product)

本筆記本使用 **NumPy** 介紹向量的兩種核心乘法：

1. **點積（內積）定義** — $\mathbf{u}\cdot\mathbf{v}$
2. **幾何意義** — 夾角、正交、投影
3. **點積性質** — 交換、分配、與範數
4. **叉積定義** — $\mathbf{u}\times\mathbf{v}$（僅 $\mathbb{R}^3$）
5. **幾何意義** — 法向量、平行四邊形面積、右手定則
6. **叉積性質** — 反交換、分配、與點積的關係
7. **純量三重積** — $(\mathbf{u}\times\mathbf{v})\cdot\mathbf{w}$ 與平行六面體體積
8. **應用** — 平面法向量、面積、體積
9. **綜合練習**

> NumPy 做的是**數值**運算（浮點近似）。需要精確分數／符號推導時請看同目錄的 `product_sympy.ipynb`。


In [1]:
import numpy as np
from IPython.display import display, Math


def _fmt_number(x):
    """Format a number for LaTeX display."""
    x = np.asarray(x).item()
    if isinstance(x, complex) or np.iscomplexobj(x):
        re, im = float(np.real(x)), float(np.imag(x))
        if abs(im) < 1e-10:
            x = re
        else:
            re_s = _fmt_number(re)
            im_s = _fmt_number(abs(im))
            sign = "+" if im >= 0 else "-"
            return f"{re_s} {sign} {im_s} i"
    x = float(np.real(x))
    if abs(x - round(x)) < 1e-10:
        return str(int(round(x)))
    return f"{x:.6g}"


def _array_to_latex(arr):
    """Convert a numpy array / scalar to a LaTeX string."""
    arr = np.asarray(arr)
    if arr.ndim == 0:
        return _fmt_number(arr)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    rows = []
    for row in arr:
        rows.append(" & ".join(_fmt_number(v) for v in row))
    body = r"\\".join(rows)
    return r"\begin{bmatrix}" + body + r"\end{bmatrix}"


def display_large(expr):
    """Display a numpy array / scalar in LaTeX with large font size."""
    display(Math(r"\large{%s}" % _array_to_latex(expr)))


def display_huge(expr):
    """Display a numpy array / scalar in LaTeX with huge font size."""
    display(Math(r"\huge{%s}" % _array_to_latex(expr)))


## 1. 點積（內積）定義

在 $\mathbb{R}^n$ 中，兩向量

$$
\mathbf{u}=\begin{bmatrix}u_1\\\vdots\\u_n\end{bmatrix},\quad
\mathbf{v}=\begin{bmatrix}v_1\\\vdots\\v_n\end{bmatrix}
$$

的**點積（dot product / inner product）**為一個**純量**：

$$
\mathbf{u}\cdot\mathbf{v}
= u_1 v_1 + u_2 v_2 + \cdots + u_n v_n
= \mathbf{u}^{\mathsf{T}}\mathbf{v}
$$

| 名稱 | 英文 | 結果 |
|------|------|------|
| **點積／內積** | dot / inner product | 純量（scalar） |

NumPy：`np.dot(u, v)`、`u @ v`（一維時），或 `np.inner(u, v)`。


In [2]:
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, -5.0, 6.0])

print("u =")
display_huge(u)
print("v =")
display_huge(v)

print("u · v =")
display_huge(np.dot(u, v))

print("via u @ v =")
display_huge(u @ v)


u =


<IPython.core.display.Math object>

v =


<IPython.core.display.Math object>

u · v =


<IPython.core.display.Math object>

via u @ v =


<IPython.core.display.Math object>

## 2. 幾何意義：夾角、正交、投影

點積與夾角 $\theta$ 的關係：

$$
\mathbf{u}\cdot\mathbf{v} = \|\mathbf{u}\|\|\mathbf{v}\|\cos\theta
\quad\Rightarrow\quad
\cos\theta = \frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|\|\mathbf{v}\|}
$$

因此：

| 條件 | 意義 |
|------|------|
| $\mathbf{u}\cdot\mathbf{v}=0$ | **正交**（垂直），$\theta=90^\circ$ |
| $\mathbf{u}\cdot\mathbf{v}>0$ | 銳角 |
| $\mathbf{u}\cdot\mathbf{v}<0$ | 鈍角 |

**範數**：`np.linalg.norm(u)`。

**投影**（$\mathbf{v}$ 投影到 $\mathbf{u}$）：

$$
\mathrm{proj}_{\mathbf{u}}\mathbf{v}
= \frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|^2}\,\mathbf{u}
$$


In [3]:
u = np.array([3.0, 0.0])
v = np.array([2.0, 2.0])

dot = np.dot(u, v)
nu = np.linalg.norm(u)
nv = np.linalg.norm(v)
cos_theta = dot / (nu * nv)
# Numerical clamp for acos domain
cos_theta = float(np.clip(cos_theta, -1.0, 1.0))
theta = np.arccos(cos_theta)

print("u =")
display_huge(u)
print("v =")
display_huge(v)
print("u · v =")
display_huge(dot)
print("||u||, ||v|| =")
display_huge(np.array([nu, nv]))
print("cos θ =")
display_huge(cos_theta)
print("θ (radians) =")
display_huge(theta)
print("θ (degrees) =")
display_huge(np.degrees(theta))

# Orthogonal check
a = np.array([1.0, 2.0, -1.0])
b = np.array([2.0, -1.0, 0.0])
print("\nOrthogonal example: a · b =")
display_huge(np.dot(a, b))
print("Orthogonal?", np.isclose(np.dot(a, b), 0))

# Projection
proj = (np.dot(u, v) / np.dot(u, u)) * u
print("\nproj_u(v) =")
display_huge(proj)
print("orthogonal remainder v - proj =")
display_huge(v - proj)
print("remainder · u =")
display_huge(np.dot(v - proj, u))


u =


<IPython.core.display.Math object>

v =


<IPython.core.display.Math object>

u · v =


...

<IPython.core.display.Math object>

orthogonal remainder v - proj =


<IPython.core.display.Math object>

remainder · u =


<IPython.core.display.Math object>

## 3. 點積的性質

對任意向量 $\mathbf{u},\mathbf{v},\mathbf{w}$ 與純量 $c$：

| 性質 | 公式 |
|------|------|
| 交換律 | $\mathbf{u}\cdot\mathbf{v}=\mathbf{v}\cdot\mathbf{u}$ |
| 分配律 | $\mathbf{u}\cdot(\mathbf{v}+\mathbf{w})=\mathbf{u}\cdot\mathbf{v}+\mathbf{u}\cdot\mathbf{w}$ |
| 純量 | $(c\mathbf{u})\cdot\mathbf{v}=c(\mathbf{u}\cdot\mathbf{v})$ |
| 正定 | $\mathbf{u}\cdot\mathbf{u}=\|\mathbf{u}\|^2\geq 0$，等號 $\Leftrightarrow\mathbf{u}=\mathbf{0}$ |
| Cauchy–Schwarz | $|\mathbf{u}\cdot\mathbf{v}|\leq\|\mathbf{u}\|\|\mathbf{v}\|$ |

注意：點積結果永遠是純量，沒有「三向量結合相乘」成向量的說法。


In [4]:
rng = np.random.default_rng(0)
u = rng.normal(size=4)
v = rng.normal(size=4)
w = rng.normal(size=4)
c = 2.5

print("Commutative? ", np.isclose(np.dot(u, v), np.dot(v, u)))
print("Distributive?", np.isclose(np.dot(u, v + w), np.dot(u, v) + np.dot(u, w)))
print("Scalar?      ", np.isclose(np.dot(c * u, v), c * np.dot(u, v)))

p = np.array([1.0, -2.0, 3.0])
q = np.array([4.0, 0.0, -1.0])
print("\n|Cauchy–Schwarz|: |p·q| =", abs(np.dot(p, q)),
      ", ||p||||q|| =", np.linalg.norm(p) * np.linalg.norm(q))
print("Holds?", abs(np.dot(p, q)) <= np.linalg.norm(p) * np.linalg.norm(q) + 1e-12)


Commutative?  True
Distributive? True
Scalar?       True

|Cauchy–Schwarz|: |p·q| = 1.0 , ||p||||q|| = 15.427248620541512
Holds? True


## 4. 叉積定義（僅 $\mathbb{R}^3$）

兩向量 $\mathbf{u},\mathbf{v}\in\mathbb{R}^3$ 的**叉積（cross product）**是另一個**向量**：

$$
\mathbf{u}\times\mathbf{v}
=
\begin{bmatrix}
u_2 v_3 - u_3 v_2 \\
u_3 v_1 - u_1 v_3 \\
u_1 v_2 - u_2 v_1
\end{bmatrix}
$$

| 名稱 | 英文 | 結果 | 空間 |
|------|------|------|------|
| **叉積** | cross product | 向量 | 主要在 $\mathbb{R}^3$ |

NumPy：`np.cross(u, v)`。


In [5]:
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, 5.0, 6.0])

print("u =")
display_huge(u)
print("v =")
display_huge(v)
print("u × v =")
display_huge(np.cross(u, v))

# Manual formula check
manual = np.array([
    u[1] * v[2] - u[2] * v[1],
    u[2] * v[0] - u[0] * v[2],
    u[0] * v[1] - u[1] * v[0],
])
print("manual formula =")
display_huge(manual)
print("Match?", np.allclose(np.cross(u, v), manual))


u =


<IPython.core.display.Math object>

v =


<IPython.core.display.Math object>

u × v =


<IPython.core.display.Math object>

manual formula =


<IPython.core.display.Math object>

Match? True


## 5. 叉積的幾何意義

$$
\|\mathbf{u}\times\mathbf{v}\| = \|\mathbf{u}\|\|\mathbf{v}\|\sin\theta
$$

等於以 $\mathbf{u},\mathbf{v}$ 為鄰邊的**平行四邊形面積**。

方向由**右手定則**決定：$\mathbf{u}\times\mathbf{v}$ **垂直**於 $\mathbf{u}$ 與 $\mathbf{v}$ 張成的平面，

$$
(\mathbf{u}\times\mathbf{v})\cdot\mathbf{u}=0,\qquad
(\mathbf{u}\times\mathbf{v})\cdot\mathbf{v}=0
$$

特殊情況：

- $\mathbf{u}\parallel\mathbf{v}$ $\Rightarrow$ $\mathbf{u}\times\mathbf{v}=\mathbf{0}$
- $\mathbf{u}\perp\mathbf{v}$ $\Rightarrow$ $\|\mathbf{u}\times\mathbf{v}\|=\|\mathbf{u}\|\|\mathbf{v}\|$


In [6]:
u = np.array([1.0, 0.0, 0.0])
v = np.array([1.0, 1.0, 0.0])
n = np.cross(u, v)

print("u =")
display_huge(u)
print("v =")
display_huge(v)
print("n = u × v =")
display_huge(n)

print("n · u =")
display_huge(np.dot(n, u))
print("n · v =")
display_huge(np.dot(n, v))

area = np.linalg.norm(n)
cos_theta = np.clip(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)), -1, 1)
sin_theta = np.sin(np.arccos(cos_theta))
print("parallelogram area ||u × v|| =")
display_huge(area)
print("||u||||v||sinθ =")
display_huge(np.linalg.norm(u) * np.linalg.norm(v) * sin_theta)

# Parallel => zero cross
a = np.array([1.0, 2.0, 3.0])
b = 5 * a
print("\nParallel: a × (5a) =")
display_huge(np.cross(a, b))


u =


<IPython.core.display.Math object>

v =


<IPython.core.display.Math object>

n = u × v =


...

<IPython.core.display.Math object>

||u||||v||sinθ =


<IPython.core.display.Math object>


Parallel: a × (5a) =


<IPython.core.display.Math object>

## 6. 叉積的性質

| 性質 | 公式 |
|------|------|
| 反交換 | $\mathbf{u}\times\mathbf{v}=-\mathbf{v}\times\mathbf{u}$ |
| 分配律 | $\mathbf{u}\times(\mathbf{v}+\mathbf{w})=\mathbf{u}\times\mathbf{v}+\mathbf{u}\times\mathbf{w}$ |
| 純量 | $(c\mathbf{u})\times\mathbf{v}=c(\mathbf{u}\times\mathbf{v})$ |
| 自叉 | $\mathbf{u}\times\mathbf{u}=\mathbf{0}$ |
| 不結合 | $(\mathbf{u}\times\mathbf{v})\times\mathbf{w}\neq\mathbf{u}\times(\mathbf{v}\times\mathbf{w})$（一般） |

Lagrange 恆等式：

$$
\|\mathbf{u}\times\mathbf{v}\|^2
= \|\mathbf{u}\|^2\|\mathbf{v}\|^2-(\mathbf{u}\cdot\mathbf{v})^2
$$


In [7]:
rng = np.random.default_rng(1)
u = rng.normal(size=3)
v = rng.normal(size=3)
w = rng.normal(size=3)
c = -1.7

print("Anti-commutative?", np.allclose(np.cross(u, v), -np.cross(v, u)))
print("Distributive?    ", np.allclose(np.cross(u, v + w), np.cross(u, v) + np.cross(u, w)))
print("Scalar?          ", np.allclose(np.cross(c * u, v), c * np.cross(u, v)))
print("Self cross zero? ", np.allclose(np.cross(u, u), 0))

# Not associative
a = np.array([1.0, 0.0, 0.0])
b = np.array([1.0, 1.0, 0.0])
cvec = np.array([0.0, 1.0, 1.0])
left = np.cross(np.cross(a, b), cvec)
right = np.cross(a, np.cross(b, cvec))
print("\n(a×b)×c =")
display_huge(left)
print("a×(b×c) =")
display_huge(right)
print("Equal?", np.allclose(left, right))

# Lagrange identity
p = np.array([2.0, -1.0, 3.0])
q = np.array([1.0, 4.0, -2.0])
lhs = np.dot(np.cross(p, q), np.cross(p, q))
rhs = np.dot(p, p) * np.dot(q, q) - np.dot(p, q) ** 2
print("\nLagrange: ||p×q||² =", lhs, ", ||p||²||q||²-(p·q)² =", rhs)
print("Match?", np.isclose(lhs, rhs))


Anti-commutative? True
Distributive?     True
Scalar?           True
Self cross zero?  True

(a×b)×c =


<IPython.core.display.Math object>

a×(b×c) =


<IPython.core.display.Math object>

Equal? False

Lagrange: ||p×q||² = 230.0 , ||p||²||q||²-(p·q)² = 230.0
Match? True


## 7. 純量三重積

$$
(\mathbf{u}\times\mathbf{v})\cdot\mathbf{w}
=
\det\begin{bmatrix}
\mathbf{u} & \mathbf{v} & \mathbf{w}
\end{bmatrix}
$$

幾何意義：以三向量為鄰邊的**平行六面體有號體積**。

- $\approx 0$ $\Leftrightarrow$ 三向量近似共面（線性相依）
- 絕對值 = 體積


In [8]:
u = np.array([1.0, 0.0, 0.0])
v = np.array([1.0, 2.0, 0.0])
w = np.array([1.0, 2.0, 3.0])

triple = np.dot(np.cross(u, v), w)
M = np.column_stack([u, v, w])

print("u, v, w as columns of M:")
display_huge(M)
print("(u × v) · w =")
display_huge(triple)
print("det(M) =")
display_huge(np.linalg.det(M))
print("Match?", np.isclose(triple, np.linalg.det(M)))

print("volume = |triple| =")
display_huge(abs(triple))

# Coplanar => zero
w_coplanar = u + 2 * v
print("\nCoplanar w = u+2v, triple =")
display_huge(np.dot(np.cross(u, v), w_coplanar))


u, v, w as columns of M:


<IPython.core.display.Math object>

(u × v) · w =


<IPython.core.display.Math object>

det(M) =


<IPython.core.display.Math object>

Match? True
volume = |triple| =


<IPython.core.display.Math object>


Coplanar w = u+2v, triple =


<IPython.core.display.Math object>

## 8. 應用：平面法向量、面積、體積

1. **平面法向量**：平面上兩方向 $\mathbf{u},\mathbf{v}$，則 $\mathbf{n}=\mathbf{u}\times\mathbf{v}$ 垂直該平面。
2. **三角形面積**：$\dfrac{1}{2}\|\mathbf{u}\times\mathbf{v}\|$
3. **平行六面體體積**：$|(\mathbf{u}\times\mathbf{v})\cdot\mathbf{w}|$


In [9]:
# Plane through P0 with directions u, v
P0 = np.array([1.0, 1.0, 1.0])
u = np.array([1.0, 0.0, -1.0])
v = np.array([0.0, 1.0, 2.0])
n = np.cross(u, v)

print("normal n = u × v =")
display_huge(n)

# Plane: n · (X - P0) = 0  =>  n·X = n·P0
rhs = np.dot(n, P0)
print(f"plane equation: {n[0]:g} x + {n[1]:g} y + {n[2]:g} z = {rhs:g}")

# Triangle area
area = 0.5 * np.linalg.norm(n)
print("triangle area =")
display_huge(area)

# Parallelepiped volume
w = np.array([2.0, -1.0, 1.0])
vol = abs(np.dot(np.cross(u, v), w))
print("parallelepiped volume =")
display_huge(vol)


normal n = u × v =


<IPython.core.display.Math object>

plane equation: 1 x + -2 y + 1 z = 0
triangle area =


<IPython.core.display.Math object>

parallelepiped volume =


<IPython.core.display.Math object>

## 9. 綜合練習

給定

$$
\mathbf{a}=\begin{bmatrix}1\\2\\-1\end{bmatrix},\quad
\mathbf{b}=\begin{bmatrix}0\\1\\3\end{bmatrix},\quad
\mathbf{c}=\begin{bmatrix}2\\-1\\1\end{bmatrix}
$$

1. 計算 $\mathbf{a}\cdot\mathbf{b}$，並求夾角 $\theta$（度）
2. 計算 $\mathbf{a}\times\mathbf{b}$，驗證它同時垂直於 $\mathbf{a}$ 與 $\mathbf{b}$
3. 求以 $\mathbf{a},\mathbf{b}$ 為鄰邊的平行四邊形面積
4. 計算純量三重積 $(\mathbf{a}\times\mathbf{b})\cdot\mathbf{c}$，並與 $\det[\mathbf{a}\ \mathbf{b}\ \mathbf{c}]$ 比對
5. 求 $\mathrm{proj}_{\mathbf{a}}\mathbf{b}$


In [10]:
a = np.array([1.0, 2.0, -1.0])
b = np.array([0.0, 1.0, 3.0])
c = np.array([2.0, -1.0, 1.0])

# 1. Dot and angle
dot = np.dot(a, b)
cos_theta = np.clip(dot / (np.linalg.norm(a) * np.linalg.norm(b)), -1, 1)
theta_deg = np.degrees(np.arccos(cos_theta))
print("1. a · b =")
display_huge(dot)
print("θ (degrees) =")
display_huge(theta_deg)

# 2. Cross and orthogonality
cross = np.cross(a, b)
print("\n2. a × b =")
display_huge(cross)
print("(a×b)·a =", np.dot(cross, a), ", (a×b)·b =", np.dot(cross, b))

# 3. Area
print("\n3. parallelogram area =")
display_huge(np.linalg.norm(cross))

# 4. Triple product
triple = np.dot(cross, c)
M = np.column_stack([a, b, c])
print("\n4. (a×b)·c =")
display_huge(triple)
print("det([a b c]) =")
display_huge(np.linalg.det(M))
print("Match?", np.isclose(triple, np.linalg.det(M)))

# 5. Projection
proj = (np.dot(a, b) / np.dot(a, a)) * a
print("\n5. proj_a(b) =")
display_huge(proj)


1. a · b =


<IPython.core.display.Math object>

θ (degrees) =


<IPython.core.display.Math object>


2. a × b =


...

<IPython.core.display.Math object>

det([a b c]) =


<IPython.core.display.Math object>

Match? True

5. proj_a(b) =


<IPython.core.display.Math object>

## 小結

| 概念 | NumPy / 公式 | 重點 |
|------|--------------|------|
| 點積 | `np.dot(u, v)` / `u @ v` | 結果是**純量** |
| 夾角 | $\cos\theta=\frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|\|\mathbf{v}\|}$ | $=0$ ⇒ 正交 |
| 投影 | $\mathrm{proj}_{\mathbf{u}}\mathbf{v}=\frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|^2}\mathbf{u}$ | 餘量垂直於 $\mathbf{u}$ |
| 叉積 | `np.cross(u, v)` | 結果是**向量**（$\mathbb{R}^3$） |
| 面積 | `np.linalg.norm(np.cross(u, v))` | 平行四邊形面積 |
| 法向 | $\mathbf{u}\times\mathbf{v}\perp\mathbf{u},\mathbf{v}$ | 右手定則 |
| 三重積 | `np.dot(np.cross(u, v), w)` / `np.linalg.det` | 平行六面體有號體積 |

**記住：**

- 點積：量「有多平行」；叉積：量「有多垂直」並給出法向
- $\mathbf{u}\cdot\mathbf{v}=0$ ⇒ 正交；$\mathbf{u}\times\mathbf{v}=\mathbf{0}$ ⇒ 平行
- 叉積反交換：$\mathbf{u}\times\mathbf{v}=-\mathbf{v}\times\mathbf{u}$
- 體積用 $|(\mathbf{u}\times\mathbf{v})\cdot\mathbf{w}|$
- 數值結果是浮點近似；精確符號推導請用 SymPy
